# Model selection with GMRQ and VAMP-2
We use cross-validation to choose the number of states of a Markov state model (MSM). Each model is scored by how well its slow eigenfunctions, estimated on training trajectories, describe the slow dynamics of *held-out* test trajectories. MSMBuilder has two such scores, and this example computes both:

* the generalized matrix Rayleigh quotient (**GMRQ**) [1], for reversible dynamics (the default `MarkovStateModel` score);
* the **VAMP-2** score [2], which does not assume reversibility (`MarkovStateModel(score_method='vamp2')`).

See the [model selection](../../gmrq.rst) page of the user guide for when to use which.

[1] McGibbon, R. T. and V. S. Pande, [Variational cross-validation of slow dynamical modes in molecular kinetics](https://doi.org/10.1063/1.4916292), J. Chem. Phys. 142, 124105 (2015).

[2] Wu, H. and F. Noé, [Variational approach for learning Markov processes from time series data](https://doi.org/10.1007/s00332-019-09567-y), J. Nonlinear Sci. 30, 23-66 (2020).

### Get Data
This example uses the doublewell dataset, which consists of ten trajectories in 1D with $x \in [-\pi, \pi]$.

In [ ]:
from msmbuilder.example_datasets import DoubleWell
trajectories = DoubleWell(random_state=0).get().trajectories
# sub-sample by taking only every 100th data point in each trajectory.
trajectories = [t[::100] for t in trajectories]
print([t.shape for t in trajectories])

### Set up pipeline
The `Pipeline` is a way of connecting together multiple estimators, so that we can create a custom model that
performs a sequence of steps. This model is relatively simple. It will first discretize the trajectory data
onto an evenly spaced grid between $-\pi$ and $\pi$, and then build an MSM.

In [ ]:
from sklearn.pipeline import Pipeline
from msmbuilder.cluster import NDGrid
from msmbuilder.msm import MarkovStateModel
import numpy as np

model = Pipeline([
    ('grid', NDGrid(min=-np.pi, max=np.pi)),
    ('msm', MarkovStateModel(n_timescales=1, lag_time=1, reversible_type='transpose', verbose=False))
])

## Cross validation 
To get an accurate indication of how well our MSMs are doing at finding the dominant eigenfunctions of our stochastic process, we need to consider the tendency of statistical models to overfit their training data. Our MSMs might build transition matrices which fit the noise in training data as opposed to the underlying signal. One way to combat overfitting in a data-efficient way is with cross validation. This example uses 5-fold cross validation.

In [ ]:
from sklearn.model_selection import KFold
n_states = [5, 10, 25, 50, 100, 200, 500, 750]
cv = KFold(n_splits=5)
results = []

for n in n_states:
    model.set_params(grid__n_bins_per_feature=n)
    for fold, (train_index, test_index) in enumerate(cv.split(trajectories)):
        train_data = [trajectories[i] for i in train_index]
        test_data = [trajectories[i] for i in test_index]

        # fit model with a subset of the data (training data),
        # then score it on this training data (which gives an
        # overly rosy picture of its performance) and on the test data.
        model.fit(train_data)
        row = {'n_states': n, 'fold': fold}
        for score in ('gmrq', 'vamp2'):
            # the score method only changes how the fitted model is scored
            model.set_params(msm__score_method=score)
            row[f'train_{score}'] = model.score(train_data)
            row[f'test_{score}'] = model.score(test_data)
        results.append(row)

### Use pandas to query our data

In [ ]:
import pandas as pd
results = pd.DataFrame(results)
results.head()

### Find the average for each fold
We use the median for its tolerance to outliers. Mean works too.

In [ ]:
avgs = (results
         .groupby('n_states')
         .aggregate('median')
         .drop('fold', axis=1))
avgs

In [ ]:
for score in ('gmrq', 'vamp2'):
    best_n = avgs[f'test_{score}'].idxmax()
    print(f'{score}: {best_n} states give the best test score, {avgs.loc[best_n, f"test_{score}"]:.4f}')

### Plot
This plot is very similar to figure 1 from [McGibbon and Pande](https://doi.org/10.1063/1.4916292). It shows
that the performance on the training set keeps going up as we increase the number of states (with the
amount of data fixed), whereas the test performance peaks and then starts going down. Both scores show
the same picture.

We should pick the model with the highest average test set performance. In this example, we're only choosing over
the number of MSMs states, but this method can also be used to evaluate the clustering method and any pre-processing
like tICA.

However, you do need to fix the number of dynamical processes to "score" (this is the `n_timescales` attribute for `MarkovStateModel`), as well as the lag time.

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, score, label in zip(axes, ('gmrq', 'vamp2'), ('GMRQ', 'VAMP-2')):
    ax.scatter(results['n_states'], results[f'train_{score}'], c='b', lw=0, alpha=0.5)
    ax.scatter(results['n_states'], results[f'test_{score}'], c='r', lw=0, alpha=0.5)
    ax.plot(avgs.index, avgs[f'test_{score}'], c='r', lw=2, label='Median test')
    ax.plot(avgs.index, avgs[f'train_{score}'], c='b', lw=2, label='Median train')
    best_n = avgs[f'test_{score}'].idxmax()
    ax.plot(best_n, avgs.loc[best_n, f'test_{score}'], c='k', marker='*', ms=15,
            ls='none', label=f'best: {best_n} states')
    ax.set_xscale('log')
    ax.set_xlabel('Number of states')
    ax.set_ylabel(f'{label} score')
    ax.set_title(label)
    ax.legend(loc='lower right', numpoints=1)
plt.tight_layout()